# Practice P4: answers

Try each problem for real before you read its answer. There's more than one right way; these are one clear way.

In [ ]:
import sys
sys.path.append("..")  # checks.py is one folder up
from checks import check

## 1. Secret messages

**How to think about it.** Work letter by letter. Turn the letter into a position 0–25, add the shift, wrap with `% 26`, and turn it back. `%` in Python always gives 0–25, even for a negative shift, so moving backwards works too. Capitals use `'A'` as the start instead of `'a'`.

In [ ]:
def secret(text, shift):
    result = []
    for c in text:
        if "a" <= c <= "z":
            start = ord("a")
        elif "A" <= c <= "Z":
            start = ord("A")
        else:
            result.append(c)      # not a letter: keep it
            continue
        position = (ord(c) - start + shift) % 26
        result.append(chr(start + position))
    return "".join(result)

In [ ]:
check(secret("Hello, World!", 3), "Khoor, Zruog!")
check(secret("xyz", 2), "zab")
check(secret("Khoor", -3), "Hello")
check(secret("abc", 29), "def")
check(secret("Tea at 4pm?", 0), "Tea at 4pm?")
check(secret("ZEBRA zebra", 1), "AFCSB afcsb")
check(secret("", 5), "")

## 2. Page ranges

**How to think about it.** Two pointers: `start` is the first page of a run, `end` walks right while the pages keep going up by one. When the run breaks, write one part, then start a new run right after it. Collecting parts in a list and joining with `","` avoids a stray comma.

In [ ]:
def page_ranges(pages):
    parts = []
    start = 0
    while start < len(pages):
        end = start
        # walk while the next page follows on
        while end + 1 < len(pages) and pages[end + 1] == pages[end] + 1:
            end += 1
        if start == end:
            parts.append(str(pages[start]))
        else:
            parts.append(str(pages[start]) + "-" + str(pages[end]))
        start = end + 1
    return ",".join(parts)

In [ ]:
check(page_ranges([1, 2, 3, 5, 7, 8, 9]), "1-3,5,7-9")
check(page_ranges([4, 6]), "4,6")
check(page_ranges([10, 11]), "10-11")
check(page_ranges([7]), "7")
check(page_ranges([]), "")
check(page_ranges([1, 3, 4, 5, 6, 20]), "1,3-6,20")
check(page_ranges([98, 99, 100, 101]), "98-101")

## 3. Mirror sentence

**How to think about it.** The palindrome check from notebook 13, with one extra step: before comparing, each pointer skips over characters that don't count. Each skip loop also checks `left < right`, so a sentence with no letters stops safely.

In [ ]:
def counts(c):
    return c.isalpha() or c.isdigit()


def is_mirror(sentence):
    left = 0
    right = len(sentence) - 1
    while left < right:
        while left < right and not counts(sentence[left]):
            left += 1
        while left < right and not counts(sentence[right]):
            right -= 1
        if sentence[left].lower() != sentence[right].lower():
            return False
        left += 1
        right -= 1
    return True

In [ ]:
check(is_mirror("Never odd or even"), True)
check(is_mirror("A man, a plan, a canal: Panama"), True)
check(is_mirror("Tea for two"), False)
check(is_mirror("No 'x' in Nixon"), True)
check(is_mirror(""), True)
check(is_mirror("?!, ."), True)
check(is_mirror("1a1"), True)
check(is_mirror("12"), False)

## 4. Quiz night leaderboard

**How to think about it.** Three rules means a key with three parts. Points go high-to-low and the others go low-to-high, so flip the points with a minus sign: `(-points, minutes, name)`. `sorted` makes a new list, so the original stays as it was.

In [ ]:
def leaderboard(teams):
    ordered = sorted(teams, key=lambda t: (-t[1], t[2], t[0]))
    return [name for name, points, minutes in ordered]

In [ ]:
teams = [("Owls", 30, 41), ("Cats", 35, 50), ("Bees", 30, 38), ("Ants", 30, 38)]
check(leaderboard(teams), ["Cats", "Ants", "Bees", "Owls"])
check(teams[0], ("Owls", 30, 41))   # the original list is not changed
check(leaderboard([]), [])
check(leaderboard([("Solo", 5, 10)]), ["Solo"])
check(leaderboard([("B", 10, 5), ("A", 10, 5), ("C", 10, 5)]), ["A", "B", "C"])
check(leaderboard([("Slow", 20, 60), ("Fast", 20, 15), ("Best", 25, 99)]), ["Best", "Fast", "Slow"])
check(leaderboard([("Zero", 0, 1), ("Minus", -5, 1)]), ["Zero", "Minus"])

## 5. Merge the meetings

**How to think about it.** Sorting by start is the key step. Then walk once: either the meeting starts after the last block ends (a new block), or it overlaps (stretch the last block's end). Use `max` for the new end, because a short meeting inside a long one must not shrink the block. Copy each pair with `[start, end]` so the input isn't changed.

In [ ]:
def busy_blocks(meetings):
    blocks = []
    for start, end in sorted(meetings, key=lambda m: m[0]):
        if not blocks or start > blocks[-1][1]:
            blocks.append([start, end])
        else:
            blocks[-1][1] = max(blocks[-1][1], end)
    return blocks

In [ ]:
check(busy_blocks([[9, 11], [14, 15], [10, 12]]), [[9, 12], [14, 15]])
check(busy_blocks([[1, 10], [2, 3], [4, 5]]), [[1, 10]])
check(busy_blocks([[1, 3], [3, 5]]), [[1, 5]])
check(busy_blocks([]), [])
check(busy_blocks([[8, 9]]), [[8, 9]])
check(busy_blocks([[5, 6], [1, 2], [3, 4]]), [[1, 2], [3, 4], [5, 6]])
check(busy_blocks([[6, 8], [1, 9], [2, 4], [4, 7]]), [[1, 9]])

## 6. Tic-tac-toe referee

**How to think about it.** Collect all 8 lines as strings: rows as they are, columns with `zip(*board)`, and the two diagonals by index. Check each line for `"XXX"` or `"OOO"`. Only if nobody won does "is there an empty square?" decide between playing and draw.

In [ ]:
def referee(board):
    lines = list(board)                                  # 3 rows
    for column in zip(*board):                           # 3 columns
        lines.append("".join(column))
    lines.append(board[0][0] + board[1][1] + board[2][2])  # diagonal ↘
    lines.append(board[0][2] + board[1][1] + board[2][0])  # diagonal ↙

    for player in ["X", "O"]:
        if player * 3 in lines:
            return player + " wins"

    if any("." in row for row in board):
        return "playing"
    return "draw"

In [ ]:
check(referee(["XOX", "OXO", "..X"]), "X wins")
check(referee(["XOX", "XOO", "OXX"]), "draw")
check(referee(["X..", ".O.", "..."]), "playing")
check(referee(["OOO", "XX.", "X.."]), "O wins")
check(referee(["XO.", "XO.", "X.."]), "X wins")
check(referee([".XO", "XO.", "O.X"]), "O wins")
check(referee(["...", "...", "..."]), "playing")
check(referee(["XXO", "OOX", "XOX"]), "draw")

## 7. Rotate the photo

**How to think about it.** New row `i` is old column `i`, read bottom to top. Reversing the rows (`photo[::-1]`) puts the bottom first, and `zip(*...)` then reads the columns out as rows. `zip` makes tuples, so convert each to a list. Slicing and `zip` both make new things, so the original isn't touched.

In [ ]:
def rotate(photo):
    upside_down = photo[::-1]
    return [list(column) for column in zip(*upside_down)]

In [ ]:
photo = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
check(rotate(photo), [[7, 4, 1], [8, 5, 2], [9, 6, 3]])
check(photo, [[1, 2, 3], [4, 5, 6], [7, 8, 9]])   # the original is not changed
check(rotate([[1, 2, 3], [4, 5, 6]]), [[4, 1], [5, 2], [6, 3]])
check(rotate([[5]]), [[5]])
check(rotate([[1, 2, 3]]), [[1], [2], [3]])
check(rotate([]), [])
check(rotate(rotate(rotate(rotate(photo)))), photo)   # four turns: back to the start

## 8. Minesweeper hints

**How to think about it.** For each empty square, count mines among the 8 neighbours. The 8 directions are every `dr, dc` in `-1, 0, 1` except `0, 0`. Check bounds before reading, as in notebook 15. Strings can't be changed, so build each new row as a list and `join` it.

In [ ]:
def count_mines(field, r, c):
    rows = len(field)
    cols = len(field[0])
    total = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr = r + dr
            nc = c + dc
            if 0 <= nr < rows and 0 <= nc < cols and field[nr][nc] == "*":
                total += 1
    return total


def hints(field):
    result = []
    for r in range(len(field)):
        row = []
        for c in range(len(field[r])):
            if field[r][c] == "*":
                row.append("*")
            else:
                row.append(str(count_mines(field, r, c)))
        result.append("".join(row))
    return result

In [ ]:
check(hints(["*...", "....", ".*..", "...."]), ["*100", "2210", "1*10", "1110"])
check(hints(["..", ".."]), ["00", "00"])
check(hints(["**", "**"]), ["**", "**"])
check(hints(["*.*"]), ["*2*"])
check(hints(["...", ".*.", "..."]), ["111", "1*1", "111"])
check(hints(["*"]), ["*"])
check(hints(["*.", "..", ".*"]), ["*1", "22", "1*"])

## 9. A calculator with no brackets

**How to think about it.** A stack holds the numbers waiting to be used. Numbers go on; an operation pops two, combines them and pushes the answer. The first pop is the right-hand number, which matters for `-` and `/`. Tell operations from numbers by checking against the four symbols, because `"-3"` is a number.

In [ ]:
def apply(left, op, right):
    if op == "+":
        return left + right
    if op == "-":
        return left - right
    if op == "*":
        return left * right
    return int(left / right)       # cut towards zero


def calculate(tokens):
    stack = []
    for token in tokens:
        if token in ["+", "-", "*", "/"]:
            right = stack.pop()
            left = stack.pop()
            stack.append(apply(left, token, right))
        else:
            stack.append(int(token))
    return stack[-1]

In [ ]:
check(calculate(["2", "1", "+", "3", "*"]), 9)
check(calculate(["4", "13", "5", "/", "+"]), 6)
check(calculate(["5"]), 5)
check(calculate(["7", "2", "-"]), 5)
check(calculate(["-7", "2", "/"]), -3)
check(calculate(["3", "-4", "*"]), -12)
check(calculate(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]), 22)

## 10. Tidy the folder path

**How to think about it.** Split on `/`, then walk the pieces with a stack of folders. Empty pieces and `.` change nothing, `..` pops (if there's anything to pop), and anything else is pushed. Joining the stack with `/` and adding a `/` in front gives the clean path; an empty stack gives `"/"`.

In [ ]:
def tidy_path(path):
    folders = []
    for piece in path.split("/"):
        if piece == "" or piece == ".":
            continue
        if piece == "..":
            if folders:
                folders.pop()
        else:
            folders.append(piece)
    return "/" + "/".join(folders)

In [ ]:
check(tidy_path("/home//tanvi/./docs/../music/"), "/home/tanvi/music")
check(tidy_path("/../"), "/")
check(tidy_path("/a/.../b"), "/a/.../b")
check(tidy_path("/"), "/")
check(tidy_path("/a/b/c/../../d"), "/a/d")
check(tidy_path("/x/./././"), "/x")
check(tidy_path("/a/../../b/../c//.//"), "/c")

## 11. ⭐ Stretch: the Game of Life

**How to think about it.** The trap is "at the same time": count everything from the old grid and write into a brand-new grid. The neighbour count is the 8-direction loop with a bounds check. Then each cell follows the three rules.

In [ ]:
def live_neighbours(grid, r, c):
    rows = len(grid)
    cols = len(grid[0])
    total = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr = r + dr
            nc = c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                total += grid[nr][nc]
    return total


def next_tick(grid):
    rows = len(grid)
    cols = len(grid[0])
    new = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            n = live_neighbours(grid, r, c)
            if grid[r][c] == 1 and (n == 2 or n == 3):
                new[r][c] = 1
            elif grid[r][c] == 0 and n == 3:
                new[r][c] = 1
    return new

In [ ]:
blinker = [[0, 0, 0], [1, 1, 1], [0, 0, 0]]
check(next_tick(blinker), [[0, 1, 0], [0, 1, 0], [0, 1, 0]])
check(blinker, [[0, 0, 0], [1, 1, 1], [0, 0, 0]])   # the original is not changed
check(next_tick(next_tick(blinker)), blinker)       # it blinks back
check(next_tick([[1, 1], [1, 1]]), [[1, 1], [1, 1]])  # a block never changes
check(next_tick([[1]]), [[0]])                        # alone: dies
check(next_tick([[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]), [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]])
check(next_tick([[1, 0, 1, 0]]), [[0, 0, 0, 0]])

## 12. ⭐ Stretch: Sudoku checker

**How to think about it.** One pass over the 81 squares. Each digit belongs to one row, one column and one box; keep a set of seen digits for each. `(r // 3, c // 3)` names the box, so a dictionary of sets keyed by that tuple works. The first repeat means `False`.

In [ ]:
def valid_sudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = {}
    for r in range(9):
        for c in range(9):
            digit = board[r][c]
            if digit == ".":
                continue
            box = (r // 3, c // 3)
            if box not in boxes:
                boxes[box] = set()
            if digit in rows[r] or digit in cols[c] or digit in boxes[box]:
                return False
            rows[r].add(digit)
            cols[c].add(digit)
            boxes[box].add(digit)
    return True

In [ ]:
good = [
    "53..7....",
    "6..195...",
    ".98....6.",
    "8...6...3",
    "4..8.3..1",
    "7...2...6",
    ".6....28.",
    "...419..5",
    "....8..79",
]
check(valid_sudoku(good), True)

bad_column = ["8" + good[0][1:]] + good[1:]          # two 8s in the first column
check(valid_sudoku(bad_column), False)

bad_row = ["53..7..5."] + good[1:]                     # two 5s in the top row
check(valid_sudoku(bad_row), False)

bad_box = good[:1] + ["6.5195..."] + good[2:]          # two 5s in the top-left box only
check(valid_sudoku(bad_box), False)

empty = ["........."] * 9
check(valid_sudoku(empty), True)

one_each = ["1........", "...2.....", "......3..", ".4.......", "....5....", ".......6.", "..7......", ".....8...", "........9"]
check(valid_sudoku(one_each), True)